In [1]:
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
import torch
import os
import json
import numpy as np
import ast


In [2]:
# --- 1. Load and Prepare the Dataset ---
df = pd.read_csv('../booksummaries/book_data_clean.csv')
print("Step 1: Loaded the Dataset...")

# Drop rows with missing essential data
df.dropna(subset=['title', 'author', 'genres', 'summary'], inplace=True)

# Safely convert the string representation of lists into actual lists
# Example: "['Sci-Fi', 'Adventure']" -> ['Sci-Fi', 'Adventure']
df['genres'] = df['genres'].apply(ast.literal_eval)

print(f"Data cleaned. Found {len(df)} clean book entries.")

Step 1: Loaded the Dataset...
Data cleaned. Found 14177 clean book entries.


In [3]:
# --- 2. Create Rich Training Examples ---
print("Creating rich training examples...")
train_examples = []
# Using a set to avoid processing the same summary multiple times if books are duplicated
processed_summaries = set()

for row in df.itertuples():
    summary = row.summary
    
    # Skip if we've already created examples for this exact summary
    if summary in processed_summaries:
        continue
    
    title = row.title
    author = row.author
    genres = row.genres
    
    # Pair 1: Structured Title/Author with Summary
    # This teaches the model that "X by Y" relates to the summary text.
    structured_input = f"{title} by {author}"
    train_examples.append(InputExample(texts=[structured_input, summary]))
    
    # Pair 2: Author with Summary
    train_examples.append(InputExample(texts=[author, summary]))

    # Pair 3: Each Genre with Summary
    # This teaches the model the semantic meaning of different genres.
    for genre in genres:
        if genre:  # Ensure genre is not an empty string
            train_examples.append(InputExample(texts=[genre, summary]))
    
    processed_summaries.add(summary)

print(f"Created {len(train_examples)} diverse training pairs.")

Creating rich training examples...
Created 57136 diverse training pairs.


In [4]:
# --- 3. Configure Model and Training ---
print("\nStep 3: Configuring Model and Training...")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("Warning: GPU not found. Training will be slow.")

# Load a pre-trained model.
model_name = 'all-mpnet-base-v2'
model = SentenceTransformer(model_name, device=device)

# Create a DataLoader to batch the training examples
# A batch size of 16 or 32 is a good starting point.
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Use MultipleNegativesRankingLoss, a great loss function for this task.
# It works by taking a batch of positive pairs and using the other items in the batch as negative examples.
train_loss = losses.MultipleNegativesRankingLoss(model)


Step 3: Configuring Model and Training...
Using device: cuda


In [5]:
# --- 4. Fine-Tune the Model ---
print("\nStep 4: Starting the Fine-Tuning Process...")

num_epochs = 2
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of train data for warm-up

model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=num_epochs,
          warmup_steps=warmup_steps,
          output_path='./artifacts/fine_tuned_book_embedder',
          show_progress_bar=True)

print(f"\nTraining complete. Model saved to './artifacts/fine_tuned_book_embedder'")


Step 4: Starting the Fine-Tuning Process...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,2.128300
1000,1.894000
1500,1.774100
2000,1.716900
2500,1.695600
3000,1.624500
3500,1.580100
4000,1.420100
4500,1.387700
5000,1.376300



Training complete. Model saved to './artifacts/fine_tuned_book_embedder'


In [6]:
# --- 5. Use the Fine-Tuned Model for Searching ---
print("\nStep 5: Using the Fine-Tuned Model for a Search Query...")

# Load your newly trained model
final = SentenceTransformer('./artifacts/fine_tuned_book_embedder')

# Embed all book summaries using the new model
# For efficiency, you should do this once and save the embeddings
book_summaries = df['summary'].tolist()
print("Encoding all book summaries with the new model... (This might take a while)")
book_embeddings = final.encode(book_summaries, convert_to_tensor=True, show_progress_bar=True)


Step 5: Using the Fine-Tuned Model for a Search Query...
Encoding all book summaries with the new model... (This might take a while)


Batches:   0%|          | 0/444 [00:00<?, ?it/s]

In [10]:
# --- 6. Save embeddings ---

os.makedirs('artifacts/embedder',exist_ok=True)
# Ensure tensor on CPU
emb_cpu = book_embeddings.detach().cpu()

meta = {
    "model_name": model_name if 'model_name' in globals() else "unknown",
    "num_items": int(emb_cpu.shape[0]),
    "dim": int(emb_cpu.shape[1]),
    "titles": df['title'].tolist(),
}

# Save the book embeddings
torch.save({"embeddings": emb_cpu, "meta": meta}, "artifacts/embedder/book_embeddings.pt")
print("Saved artifacts/embedder/book_embeddings.pt")

Saved artifacts/embedder/book_embeddings.pt
